In [3]:
# ============================================================
# Chaotic Time Series Generators
# ============================================================
# Each function generates a 1D or multi-dimensional chaotic
# time series, discards transients, and returns a clean array.
# At the bottom: save any of them to CSV with one call.
# ============================================================

import numpy as np
import os


# ----------------------------------------------------------
# 1. LOGISTIC MAP
# ----------------------------------------------------------
# x(t+1) = r * x(t) * (1 - x(t))
#
# The simplest chaotic system. One-dimensional discrete map.
# Chaotic for r > 3.57 (e.g., r = 4.0 is fully chaotic).
# State stays in [0, 1].
# ----------------------------------------------------------
def logistic_map(n_steps, r=4.0, x0=0.4, transient=1000):
    """
    Parameters
    ----------
    n_steps   : int   - number of data points to return
    r         : float - control parameter (chaos for r > 3.57)
    x0        : float - initial condition in (0, 1)
    transient : int   - steps to discard

    Returns
    -------
    x : ndarray of shape (n_steps,)
    """
    total = n_steps + transient
    x = np.zeros(total)
    x[0] = x0
    for t in range(total - 1):
        x[t + 1] = r * x[t] * (1.0 - x[t])
    return x[transient:]


# ----------------------------------------------------------
# 2. HÉNON MAP
# ----------------------------------------------------------
# x(t+1) = 1 - a*x(t)^2 + y(t)
# y(t+1) = b*x(t)
#
# Classic 2D discrete chaotic map. The standard parameters
# a=1.4, b=0.3 produce a chaotic attractor.
# ----------------------------------------------------------
def henon_map(n_steps, a=1.4, b=0.3, x0=0.1, y0=0.1, transient=1000):
    """
    Returns
    -------
    xy : ndarray of shape (n_steps, 2) — columns are [x, y]
    """
    total = n_steps + transient
    x = np.zeros(total)
    y = np.zeros(total)
    x[0], y[0] = x0, y0
    for t in range(total - 1):
        x[t + 1] = 1.0 - a * x[t] ** 2 + y[t]
        y[t + 1] = b * x[t]
    return np.column_stack([x[transient:], y[transient:]])


# ----------------------------------------------------------
# 3. RULKOV MAP
# ----------------------------------------------------------
# u(t+1) = beta / (1 + u(t)^2) + v(t)
# v(t+1) = v(t) - nu*u(t) - sigma
#
# Models neuron membrane potential (u) and slow ion dynamics (v).
# Exhibits chaotic bursting for beta=4.1, nu=sigma=0.001.
# This is the map used in the Topal & Eroglu (2023) paper.
# ----------------------------------------------------------
def rulkov_map(n_steps, beta=4.1, nu=0.001, sigma=0.001,
               u0=0.1, v0=-2.9, transient=5000):
    """
    Returns
    -------
    uv : ndarray of shape (n_steps, 2) — columns are [u, v]
    """
    total = n_steps + transient
    u = np.zeros(total)
    v = np.zeros(total)
    u[0], v[0] = u0, v0
    for t in range(total - 1):
        u[t + 1] = beta / (1.0 + u[t] ** 2) + v[t]
        v[t + 1] = v[t] - nu * u[t] - sigma
    return np.column_stack([u[transient:], v[transient:]])


# ----------------------------------------------------------
# 4. TINKERBELL MAP
# ----------------------------------------------------------
# x(t+1) = x(t)^2 - y(t)^2 + a*x(t) + b*y(t)
# y(t+1) = 2*x(t)*y(t) + c*x(t) + d*y(t)
#
# 2D discrete chaotic map with a complex attractor shape.
# Also used in Topal & Eroglu (2023) supplemental material.
# ----------------------------------------------------------
def tinkerbell_map(n_steps, a=0.9, b=-0.6013, c=2.0, d=0.5,
                   x0=-0.72, y0=-0.64, transient=5000):
    """
    Returns
    -------
    xy : ndarray of shape (n_steps, 2) — columns are [x, y]
    """
    total = n_steps + transient
    x = np.zeros(total)
    y = np.zeros(total)
    x[0], y[0] = x0, y0
    for t in range(total - 1):
        x[t + 1] = x[t] ** 2 - y[t] ** 2 + a * x[t] + b * y[t]
        y[t + 1] = 2.0 * x[t] * y[t] + c * x[t] + d * y[t]
    return np.column_stack([x[transient:], y[transient:]])


# ----------------------------------------------------------
# 5. LORENZ SYSTEM (continuous, integrated with RK4)
# ----------------------------------------------------------
# dx/dt = sigma * (y - x)
# dy/dt = x * (rho - z) - y
# dz/dt = x * y - beta * z
#
# The iconic 3D continuous chaotic system. We integrate it
# with 4th-order Runge-Kutta and sample at fixed intervals.
# Standard parameters: sigma=10, rho=28, beta=8/3.
# ----------------------------------------------------------
def lorenz_system(n_steps, sigma=10.0, rho=28.0, beta=8.0/3.0,
                  dt=0.01, sample_every=1,
                  x0=1.0, y0=1.0, z0=1.0, transient=5000):
    """
    Parameters
    ----------
    dt           : float - integration time step
    sample_every : int   - save every N-th integration step
                           (effective sampling dt_eff = dt * sample_every)

    Returns
    -------
    xyz : ndarray of shape (n_steps, 3) — columns are [x, y, z]
    """
    total = (n_steps + transient) * sample_every

    def rhs(state):
        x, y, z = state
        return np.array([
            sigma * (y - x),
            x * (rho - z) - y,
            x * y - beta * z
        ])

    def rk4_step(state, dt):
        k1 = rhs(state)
        k2 = rhs(state + 0.5 * dt * k1)
        k3 = rhs(state + 0.5 * dt * k2)
        k4 = rhs(state + dt * k3)
        return state + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

    state = np.array([x0, y0, z0])
    collected = []

    for t in range(total):
        state = rk4_step(state, dt)
        if t % sample_every == 0:
            collected.append(state.copy())

    collected = np.array(collected)
    return collected[transient:]


# ----------------------------------------------------------
# 6. RÖSSLER SYSTEM (continuous, integrated with RK4)
# ----------------------------------------------------------
# dx/dt = -y - z
# dy/dt = x + a*y
# dz/dt = b + z*(x - c)
#
# Another classic 3D chaotic flow. Simpler attractor than
# Lorenz — a single folded band. Standard: a=0.2, b=0.2, c=5.7.
# ----------------------------------------------------------
def rossler_system(n_steps, a=0.2, b=0.2, c=5.7,
                   dt=0.01, sample_every=1,
                   x0=1.0, y0=1.0, z0=1.0, transient=5000):
    """
    Returns
    -------
    xyz : ndarray of shape (n_steps, 3) — columns are [x, y, z]
    """
    total = (n_steps + transient) * sample_every

    def rhs(state):
        x, y, z = state
        return np.array([
            -y - z,
            x + a * y,
            b + z * (x - c)
        ])

    def rk4_step(state, dt):
        k1 = rhs(state)
        k2 = rhs(state + 0.5 * dt * k1)
        k3 = rhs(state + 0.5 * dt * k2)
        k4 = rhs(state + dt * k3)
        return state + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

    state = np.array([x0, y0, z0])
    collected = []

    for t in range(total):
        state = rk4_step(state, dt)
        if t % sample_every == 0:
            collected.append(state.copy())

    collected = np.array(collected)
    return collected[transient:]


# ----------------------------------------------------------
# 7. MACKEY-GLASS DELAY EQUATION
# ----------------------------------------------------------
# dx/dt = beta * x(t-tau) / (1 + x(t-tau)^n) - gamma * x(t)
#
# Delay differential equation. Chaotic for tau > 17.
# Produces a scalar time series (1D).
# Standard benchmark in reservoir computing literature.
# ----------------------------------------------------------
def mackey_glass(n_steps, tau=17, beta=0.2, gamma=0.1, n=10,
                 dt=1.0, x0=1.2, transient=2000):
    """
    Returns
    -------
    x : ndarray of shape (n_steps,)
    """
    total = n_steps + transient
    history = np.ones(tau + 1) * x0
    x_t = x0
    series = np.zeros(total)

    for t in range(total):
        x_delayed = history[0]
        dx = beta * x_delayed / (1.0 + x_delayed ** n) - gamma * x_t
        x_t = x_t + dt * dx
        history = np.roll(history, -1)
        history[-1] = x_t
        series[t] = x_t

    return series[transient:]


# ----------------------------------------------------------
# 8. CHUA'S CIRCUIT
# ----------------------------------------------------------
# dx/dt = alpha * (y - x - h(x))
# dy/dt = x - y + z
# dz/dt = -beta_c * y
#
# where h(x) = m1*x + 0.5*(m0-m1)*(|x+1| - |x-1|)
#
# Electronic circuit that produces chaos. One of the simplest
# physical systems exhibiting a double-scroll attractor.
# ----------------------------------------------------------
def chua_circuit(n_steps, alpha=15.6, beta_c=28.0,
                 m0=-1.143, m1=-0.714,
                 dt=0.001, sample_every=10,
                 x0=0.7, y0=0.0, z0=0.0, transient=5000):
    """
    Returns
    -------
    xyz : ndarray of shape (n_steps, 3) — columns are [x, y, z]
    """
    total = (n_steps + transient) * sample_every

    def h(x):
        return m1 * x + 0.5 * (m0 - m1) * (np.abs(x + 1) - np.abs(x - 1))

    def rhs(state):
        x, y, z = state
        return np.array([
            alpha * (y - x - h(x)),
            x - y + z,
            -beta_c * y
        ])

    def rk4_step(state, dt):
        k1 = rhs(state)
        k2 = rhs(state + 0.5 * dt * k1)
        k3 = rhs(state + 0.5 * dt * k2)
        k4 = rhs(state + dt * k3)
        return state + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

    state = np.array([x0, y0, z0])
    collected = []

    for t in range(total):
        state = rk4_step(state, dt)
        if t % sample_every == 0:
            collected.append(state.copy())

    collected = np.array(collected)
    return collected[transient:]


# ----------------------------------------------------------
# 9. TENT MAP
# ----------------------------------------------------------
# x(t+1) = mu * min(x(t), 1 - x(t))
#
# Piecewise linear 1D map. Chaotic for mu = 2.
# Simpler than logistic map but fully chaotic — useful as
# a baseline or sanity check.
# ----------------------------------------------------------
def tent_map(n_steps, mu=1.99, x0=0.4, transient=1000):
    """
    Returns
    -------
    x : ndarray of shape (n_steps,)
    """
    total = n_steps + transient
    x = np.zeros(total)
    x[0] = x0
    for t in range(total - 1):
        x[t + 1] = mu * min(x[t], 1.0 - x[t])
    return x[transient:]


# ==========================================================
# SAVE TO CSV
# ==========================================================
def save_timeseries(data, filename, output_dir="../data/chaotic_data"):
    """
    Save a time series array to CSV.

    Parameters
    ----------
    data       : ndarray of shape (T,) or (T, d)
    filename   : str — e.g. "lorenz.csv"
    output_dir : str — folder to save in
    """
    os.makedirs(output_dir, exist_ok=True)
    filepath = os.path.join(output_dir, filename)
    np.savetxt(filepath, data, delimiter=",", fmt="%.10f")
    shape_str = f"({data.shape[0]},)" if data.ndim == 1 else str(data.shape)
    print(f"  Saved {filepath}  shape={shape_str}")

In [4]:
# ==========================================================
# GENERATE AND SAVE ALL SYSTEMS
# ==========================================================
N = 10000  # number of data points per system

print("Generating chaotic time series...\n")

# 1D maps
save_timeseries(logistic_map(N), "logistic_map.csv")
save_timeseries(tent_map(N), "tent_map.csv")
save_timeseries(mackey_glass(N), "mackey_glass.csv")

# 2D maps
save_timeseries(henon_map(N), "henon_map.csv")
save_timeseries(rulkov_map(N), "rulkov_map.csv")
save_timeseries(tinkerbell_map(N), "tinkerbell_map.csv")

# 3D continuous flows
save_timeseries(lorenz_system(N), "lorenz_system.csv")
save_timeseries(rossler_system(N), "rossler_system.csv")
save_timeseries(chua_circuit(N), "chua_circuit.csv")

print("\nDone! All files saved to chaotic_data/")
print("\nTo load in your notebook:")
print("  data = np.loadtxt('chaotic_data/lorenz_system.csv', delimiter=',')")
print("  x_component = data[:, 0].reshape(-1, 1)  # use one component for ESN")

Generating chaotic time series...

  Saved ../data/chaotic_data/logistic_map.csv  shape=(10000,)
  Saved ../data/chaotic_data/tent_map.csv  shape=(10000,)
  Saved ../data/chaotic_data/mackey_glass.csv  shape=(10000,)
  Saved ../data/chaotic_data/henon_map.csv  shape=(10000, 2)
  Saved ../data/chaotic_data/rulkov_map.csv  shape=(10000, 2)
  Saved ../data/chaotic_data/tinkerbell_map.csv  shape=(10000, 2)
  Saved ../data/chaotic_data/lorenz_system.csv  shape=(10000, 3)
  Saved ../data/chaotic_data/rossler_system.csv  shape=(10000, 3)
  Saved ../data/chaotic_data/chua_circuit.csv  shape=(10000, 3)

Done! All files saved to chaotic_data/

To load in your notebook:
  data = np.loadtxt('chaotic_data/lorenz_system.csv', delimiter=',')
  x_component = data[:, 0].reshape(-1, 1)  # use one component for ESN
